# Contoso HR Policy RAG — LangChain Notebook

This notebook adapts the Python application into an interactive, step-by-step workflow.

**Pipeline:** Retrieve → Augment → Generate

- **Retrieve:** Create a query embedding with LangChain and run vector search in Azure AI Search.
- **Augment:** Build the prompt with `ChatPromptTemplate`.
- **Generate:** Invoke the LangChain chat model and parse the response.

The notebook uses your existing Azure AI Foundry / Azure OpenAI-compatible endpoint and Azure AI Search index. Keep your `.env` file in the same directory as this notebook.


## 1. Install dependencies

Run this cell once in the notebook's Python environment.

In [1]:
%pip install -q langchain-core langchain-openai openai azure-search-documents azure-core python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Load configuration

Create a `.env` file in the notebook's working directory with these variable names:

- `AZURE_AI_FOUNDRY_ENDPOINT`
- `AZURE_AI_FOUNDRY_API_KEY`
- `AZURE_OPENAI_EMBEDDING_DEPLOYMENT`
- `AZURE_OPENAI_CHAT_DEPLOYMENT`
- `AZURE_SEARCH_ENDPOINT`
- `AZURE_SEARCH_API_KEY`
- `AZURE_SEARCH_INDEX`

Do not paste API keys into notebook cells or commit the `.env` file to source control.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

REQUIRED_VARIABLES = [
    "AZURE_AI_FOUNDRY_ENDPOINT",
    "AZURE_AI_FOUNDRY_API_KEY",
    "AZURE_OPENAI_EMBEDDING_DEPLOYMENT",
    "AZURE_OPENAI_CHAT_DEPLOYMENT",
    "AZURE_SEARCH_ENDPOINT",
    "AZURE_SEARCH_API_KEY",
    "AZURE_SEARCH_INDEX",
]

missing = [name for name in REQUIRED_VARIABLES if not os.getenv(name)]
if missing:
    raise RuntimeError(
        "Missing environment variables: " + ", ".join(missing)
        + ". Check your .env file."
    )

print("Configuration loaded. Secret values are not displayed.")


Configuration loaded. Secret values are not displayed.


## 3. Define Azure AI Search fields and create clients

The index field names below match the original application.

In [2]:
from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.models import VectorizedQuery

KEY_FIELD = "chunk_id"
CONTENT_FIELD = "chunk"
TITLE_FIELD = "title"
VECTOR_FIELD = "text_vector"

endpoint = os.environ["AZURE_AI_FOUNDRY_ENDPOINT"].rstrip("/") + "/"
api_key = os.environ["AZURE_AI_FOUNDRY_API_KEY"]

embeddings = OpenAIEmbeddings(
    model=os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"],
    base_url=endpoint,
    api_key=api_key,
)

chat_model = ChatOpenAI(
    model=os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"],
    base_url=endpoint,
    api_key=api_key,
    temperature=1,
)

search_client = SearchClient(
    endpoint=os.environ["AZURE_SEARCH_ENDPOINT"],
    index_name=os.environ["AZURE_SEARCH_INDEX"],
    credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
)

print("LangChain embedding model, chat model, and Azure AI Search client initialized.")


LangChain embedding model, chat model, and Azure AI Search client initialized.


## 4. Inspect the Azure AI Search index (optional)

This helps confirm that the configured index contains the expected fields.

In [3]:
index_client = SearchIndexClient(
    endpoint=os.environ["AZURE_SEARCH_ENDPOINT"],
    credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
)

index = index_client.get_index(os.environ["AZURE_SEARCH_INDEX"])
print(f"Fields in index '{index.name}':")
for field in index.fields:
    print(f"  - {field.name} | type={field.type} | searchable={field.searchable}")


Fields in index 'hr-policy-index':
  - chunk_id | type=Edm.String | searchable=True
  - parent_id | type=Edm.String | searchable=False
  - chunk | type=Edm.String | searchable=True
  - title | type=Edm.String | searchable=True
  - text_vector | type=Collection(Edm.Single) | searchable=True


## 5. Step 1 — Retrieve

Embed the user's question, then use that vector to search Azure AI Search.

In [4]:
def retrieve(user_question: str, k: int = 3):
    print("Generating query embedding...")
    query_vector = embeddings.embed_query(user_question)
    print(f"Embedding generated ({len(query_vector)} dimensions)")

    vector_query = VectorizedQuery(
        vector=query_vector,
        k_nearest_neighbors=k,
        fields=VECTOR_FIELD,
    )

    print("Searching Azure AI Search...")
    results = search_client.search(
        search_text=None,
        vector_queries=[vector_query],
        select=[KEY_FIELD, TITLE_FIELD, CONTENT_FIELD],
    )
    retrieved = list(results)

    print(f"Retrieved {len(retrieved)} chunks:")
    for result in retrieved:
        key = result.get(KEY_FIELD, "Unknown")
        title = result.get(TITLE_FIELD, "Untitled")
        content = result.get(CONTENT_FIELD, "")
        print(f"\n[{key}]\nTitle: {title}\nContent: {content[:500]}...")

    return retrieved


## 6. Step 2 — Augment

Build a context string from the retrieved chunks and use a LangChain prompt template.

In [5]:
def build_context(retrieved):
    context_parts = []
    for result in retrieved:
        title = result.get(TITLE_FIELD, "Untitled")
        content = result.get(CONTENT_FIELD, "")
        context_parts.append(f"[Policy: {title}]\n{content}")
    return "\n\n".join(context_parts)


prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are the Contoso India HR Policy Assistant.
Answer the user's question using ONLY the information in the supplied context.

Rules:
1. Do not use outside knowledge.
2. Do not make up information.
3. If the answer is not present in the context, say that you don't have that information.
4. Mention the policy name/source used for the answer.
5. Give a concise and clear answer.
6. If multiple policies are relevant, mention each relevant policy.""",
    ),
    (
        "human",
        "Context:\n-------------------------\n{context}\n"
        "-------------------------\nQuestion:\n{question}",
    ),
])

context = build_context(retrieved) if "retrieved" in globals() else ""
print("Prompt template created.")
print("Retrieved context is ready." if context else "Run the Retrieve cell and assign its result to `retrieved` before generating.")


Prompt template created.
Run the Retrieve cell and assign its result to `retrieved` before generating.


## 7. Step 3 — Generate

LangChain Expression Language (LCEL) connects the prompt, chat model, and string output parser.

In [6]:
rag_chain = prompt | chat_model | StrOutputParser()
print("RAG chain created: prompt -> chat model -> string output parser.")


RAG chain created: prompt -> chat model -> string output parser.


## 8. Ask a question

Edit the question below, then run this cell. It runs the full Retrieve → Augment → Generate pipeline.

In [10]:
# user_question = "What is the process of applying for leave?"
# user_question = "How many days of paternity leave are allowed?"
user_question = "Who is prime minister of India?"

if not user_question.strip():
    raise ValueError("Question cannot be empty.")

retrieved = retrieve(user_question, k=3)
context = build_context(retrieved)

print("\nGenerating answer...\n")
answer = rag_chain.invoke({
    "context": context,
    "question": user_question,
})

print("=" * 70)
print("ANSWER")
print("=" * 70)
print(answer)

print("\n" + "=" * 70)
print("SOURCES")
print("=" * 70)
if retrieved:
    for result in retrieved:
        print(
            f"- {result.get(TITLE_FIELD, 'Untitled')} "
            f"({result.get(KEY_FIELD, 'Unknown')})"
        )
else:
    print("- No documents were retrieved.")


Generating query embedding...
Embedding generated (1536 dimensions)
Searching Azure AI Search...
Retrieved 3 chunks:

[b9b262b6feee_aHR0cHM6Ly9zdG9yYWdlYWNjMTA1OGEuYmxvYi5jb3JlLndpbmRvd3MubmV0L3BvbGljeS9TaWNrX0xlYXZlX2FuZF9NZWRpY2FsX0JlbmVmaXRzX1BvbGljeS5kb2N40_pages_0]
Title: Sick_Leave_and_Medical_Benefits_Policy.docx
Content: CONTOSO INDIA PRIVATE LIMITED
Sick Leave & Medical Benefits Policy
	POLICY NO.
HR-005
	EFFECTIVE DATE
January 1, 2026
	POLICY OWNER
People & Culture Team



1. Purpose
This policy explains paid sick leave and medical benefits at Contoso India, in line with the applicable state Shops & Establishments Act and the Employees' State Insurance (ESI) Act, 1948, where it applies to you.
2. Scope
This policy applies to all confirmed full-time and part-time employees at Contoso India.
3. Sick Leave Entitl...

[b9b262b6feee_aHR0cHM6Ly9zdG9yYWdlYWNjMTA1OGEuYmxvYi5jb3JlLndpbmRvd3MubmV0L3BvbGljeS9SZW1vdGVfV29ya19Qb2xpY3kuZG9jeA2_pages_0]
Title: Remote_Work_Policy.docx
Conten

## 9. Try another question

Change `user_question` in the previous cell and run it again.

Example questions:
- What is the annual leave policy?
- How many days of paternity leave are available?
- What is the process for applying for leave?

The answers depend on the documents indexed in your Azure AI Search index. If the retrieved context does not contain the answer, the prompt instructs the model to say that it does not have the information.

## Troubleshooting

- **Authentication or deployment error:** Verify the endpoint, key, and deployment names in `.env`.
- **Azure AI Search error:** Verify the search endpoint, key, index name, and field names.
- **No relevant chunks:** Check that the index contains documents and that `text_vector` uses embeddings compatible with the configured embedding deployment.
- **Embedding dimension mismatch:** The query embedding dimensions must match the vector field's configured dimensions.
- **Endpoint compatibility:** This notebook assumes the configured endpoint supports the OpenAI-compatible API used by `langchain-openai`. If your Azure AI Foundry endpoint requires a different URL or authentication configuration, adjust the client initialization accordingly.

**Security:** If credentials in a previously shared `.env` file were real, rotate those keys and update the local file before running this notebook.
